## Imports

In [1]:
from pyspark.sql import SparkSession, functions as F


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/30 14:47:15 WARN Utils: Your hostname, BDSP, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/30 14:47:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 14:47:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 14:47:30 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet.
java.io.FileNotFoundException: File does not exist: hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet
	at org.apache.hadoop.hdfs.Distribute

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



[Stage 1:>                                                          (0 + 4) / 4]

linhas: 9554778


## Notebook 00 – Inspeção e limpeza dos dados brutos

Este notebook é o primeiro passo do pipeline do CA1. Ele carrega três meses de registros de corridas do NYC Yellow Taxi (janeiro a março de 2024) a partir do HDFS, mede a precisão nativa das colunas que depois serão degradadas, verifica a qualidade dos dados e grava uma base limpa única (L0) de volta no HDFS. Todos os formatos de armazenamento e níveis de precisão dos próximos notebooks são derivados dessa base, então todos os experimentos comparam exatamente o mesmo conjunto de corridas.

## Licence and LGDP Rules  
Os arquivos brutos são publicados pela NYC Taxi and Limousine Commission em formato Parquet (NYC Taxi and Limousine Commission, n.d.). As versões em CSV usadas mais adiante no projeto são geradas a partir desses arquivos.

Os dados são os registros do Yellow Taxi de janeiro a março de 2024, publicados em Parquet pela NYC Taxi and Limousine Commission e baixados em 29 de setembro de 2026, junto com a tabela de zonas `taxi_zone_lookup.csv` (NYC Taxi and Limousine Commission, n.d.b).

Não há licença nomeada. A Lei de Open Data de Nova York garante uso sem restrições, exigindo apenas que quem redistribui identifique a fonte e a versão e descreva as modificações. As modificações feitas aqui estão documentadas nas seções 5 e seguintes. Os dados são fornecidos "as is", e o TLC não garante sua exatidão, O dataset não tem identificadores de motorista, veículo ou passageiro, então não há dados pessoais.

## SETUP & ENVORMENT

Os dados são lidos com PySpark a partir do HDFS rodando em modo pseudo-distribuído, que é o ambiente de processamento usado em todo o projeto. A sessão do Spark usa os 4 núcleos locais e 4 GB de memória para o driver, que é o limite da máquina virtual.

O fuso horário da sessão é fixado em UTC. Pickup e dropoff são armazenados como timestamps sem fuso horário, e se o Spark usasse o fuso local da VM (Europe/Dublin), a mudança de horário de verão em 31 de março de 2024 poderia deslocar em uma hora a duração calculada de algumas corridas.

In [15]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[4]").appName("ca1-00-inspect")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())

df = spark.read.parquet("hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet")
df.printSchema()
print("Checagem de Linhas:", df.count())

26/09/30 15:49:08 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet.
java.io.FileNotFoundException: File does not exist: hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSo

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

Checagem de Linhas: 9554778


Os três arquivos mensais têm o mesmo schema, então podem ser lidos juntos com um único caminho com curinga. O dataset combinado tem 9.554.778 corridas e 19 colunas. Pickup e dropoff são lidos como `timestamp_ntz`, distância e valores monetários como `double`, e os IDs de localização como `integer`.

In [16]:
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
df.select(F.min("tpep_pickup_datetime").alias("pickup_min"),
          F.max("tpep_pickup_datetime").alias("pickup_max"),
          F.avg((F.col("trip_distance") <= 0).cast("int")).alias("dist_zero_ou_neg"),
          F.avg((dur <= 0).cast("int")).alias("dur_zero_ou_neg"),
          F.avg((dur > 4 * 3600).cast("int")).alias("dur_maior_4h"),
          F.expr("percentile_approx(trip_distance, array(0.5, 0.99))").alias("dist_p50_p99")
         ).show(vertical=True, truncate=False)

[Stage 76:===========================================>              (3 + 1) / 4]

-RECORD 0---------------------------------
 pickup_min       | 2002-12-31 22:17:10   
 pickup_max       | 2024-04-01 00:34:55   
 dist_zero_ou_neg | 0.022581791015971277  
 dur_zero_ou_neg  | 2.9315176134913864E-4 
 dur_maior_4h     | 5.977114277275726E-4  
 dist_p50_p99     | [1.7, 19.87]          



Aqui podemos ver que a mediana dos valores esta diferente do que imaginamos ( consultar literatura ) 

In [17]:
# Distancia media - Medicao de numeros nulos 

cols = ["passenger_count", "RatecodeID", "PULocationID", "DOLocationID", "trip_distance"]
df.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in cols]).show(vertical=True)

df.select(F.max("trip_distance").alias("dist_max"),
          F.avg(F.col("PULocationID").isin(264, 265).cast("int")).alias("pu_desconhecido"),
          F.avg(F.col("DOLocationID").isin(264, 265).cast("int")).alias("do_desconhecido")
         ).show(vertical=True)

zones = spark.read.option("header", True).csv("hdfs://localhost:9000/ca1/raw/taxi_zone_lookup.csv")
zones.filter(F.col("LocationID").cast("int") >= 264).show(truncate=False)

-RECORD 0------------------------------
 passenger_count | 0.07870010166641235 
 RatecodeID      | 0.07870010166641235 
 PULocationID    | 0.0                 
 DOLocationID    | 0.0                 
 trip_distance   | 0.0                 

-RECORD 0-------------------------------
 dist_max        | 312722.3             
 pu_desconhecido | 0.003847289806210... 
 do_desconhecido | 0.008883408908087661 

+----------+-------+--------------+------------+
|LocationID|Borough|Zone          |service_zone|
+----------+-------+--------------+------------+
|264       |Unknown|N/A           |N/A         |
|265       |N/A    |Outside of NYC|N/A         |
+----------+-------+--------------+------------+



Aqui usamos a distancia media para medir a quantidade real de numeros nulos 

Dado a natureza do problema, as features escolhidas foram : distância, hora do dia e dia da semana extraídos do pickup, as zonas de embarque e desembarque. 

In [18]:
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
t_ini = F.lit("2024-01-01 00:00:00").cast("timestamp_ntz")
t_fim = F.lit("2024-04-01 00:00:00").cast("timestamp_ntz")

filtros = [
    ("periodo jan-mar 2024", (F.col("tpep_pickup_datetime") >= t_ini) & (F.col("tpep_pickup_datetime") < t_fim)),
    ("distancia (0, 100] mi", (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)),
    ("duracao [60 s, 4 h]", (dur >= 60) & (dur <= 4 * 3600)),
    ("velocidade <= 80 mph", F.col("trip_distance") / (dur / 3600) <= 80),
    ("zonas conhecidas", ~F.col("PULocationID").isin(264, 265) & ~F.col("DOLocationID").isin(264, 265)),
]

atual = df
n_ant = df.count()
print(f"{'inicio':<25}{n_ant:>12,}")
for nome, cond in filtros:
    atual = atual.filter(cond)
    n = atual.count()
    print(f"{nome:<25}{n:>12,}   removidas: {n_ant - n:>9,} ({(n_ant - n) / n_ant:.2%})")
    n_ant = n

inicio                      9,554,778
periodo jan-mar 2024        9,554,757   removidas:        21 (0.00%)
distancia (0, 100] mi       9,338,741   removidas:   216,016 (2.26%)
duracao [60 s, 4 h]         9,298,539   removidas:    40,202 (0.43%)
velocidade <= 80 mph        9,298,271   removidas:       268 (0.00%)
zonas conhecidas            9,213,529   removidas:    84,742 (0.91%)


Gravar dados no HDFS 

In [ ]:
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
clean = atual.withColumn("duration_s_true", dur)
clean.write.mode("overwrite").parquet("hdfs://localhost:9000/ca1/clean/yellow_2024q1")

chk = spark.read.parquet("hdfs://localhost:9000/ca1/clean/yellow_2024q1")
print("linhas:", chk.count())
chk.select(F.min("duration_s_true").alias("dur_min"),
           F.max("duration_s_true").alias("dur_max"),
           F.min("tpep_pickup_datetime").alias("pickup_min"),
           F.max("tpep_pickup_datetime").alias("pickup_max")).show(vertical=True)

Apos o carregamento, podemos notar que tivemos : 9213529 linhas ( equivalente a 92% de todos os dados baixados inicialmente). Essa perda de 8% nao iria interferir nos proximos passos do projeto.  